In [0]:


#  Raw Layer Ingestion


dbutils.widgets.text("resource_type", "Patient")
dbutils.widgets.text("backfill_days", "3")
resource_type = dbutils.widgets.get("resource_type")
backfill_days = int(dbutils.widgets.get("backfill_days"))


import sys, os, uuid, json
from datetime import datetime, timezone, timedelta
sys.path.append(os.path.abspath("../common"))
from config import FHIR_BASE_URL, PAGE_SIZE, WATERMARK_TABLE, RUN_LOG_TABLE, raw_path
from fhir_client import FhirClient

run_id = str(uuid.uuid4())
run_started_at = datetime.now(timezone.utc)
run_date = run_started_at.strftime("%Y-%m-%d")


watermark_row = spark.sql(f"""
    SELECT last_updated_watermark
    FROM {WATERMARK_TABLE}
    WHERE resource_type = '{resource_type}'
""").collect()

existing_watermark = watermark_row[0]["last_updated_watermark"] if watermark_row else None

if existing_watermark:
    since_iso = existing_watermark
else:
    since_iso = (run_started_at - timedelta(days=backfill_days)).strftime("%Y-%m-%dT%H:%M:%SZ")

print(f"[{resource_type}] since={since_iso}")

# COMMAND ----------
client = FhirClient(FHIR_BASE_URL, page_size=PAGE_SIZE)
now_iso = run_started_at.strftime("%Y-%m-%dT%H:%M:%SZ")

try:
    result = client.fetch_incremental(resource_type, since_iso=since_iso, now_iso=now_iso)
except Exception as e:
    spark.sql(f"""
        UPDATE {WATERMARK_TABLE}
        SET last_run_started_at = TIMESTAMP('{run_started_at.isoformat()}'),
            last_run_finished_at = current_timestamp(),
            status = 'FAILED'
        WHERE resource_type = '{resource_type}'
    """)
    raise

# COMMAND ----------
# MAGIC %md ### Persist raw pages to the Volume, untouched
# MAGIC Folder structure: `/Volumes/fhir_lakehouse/raw/raw_data/<Resource>/<date>/page_N.json`

# COMMAND ----------
out_dir = raw_path(resource_type, run_date)
dbutils.fs.mkdirs(out_dir)

log_rows = []
for page in result.pages:
    file_path = f"{out_dir}/page_{page.page_number:04d}.json"
    dbutils.fs.put(file_path, page.raw_json, overwrite=True)
    log_rows.append((
        run_id, resource_type, "RAW", page.request_url,
        page.fetched_at_iso, page.page_number, page.entry_count, "SUCCESS", None
    ))

print(f"[{resource_type}] wrote {len(result.pages)} page(s), "
      f"{result.total_entries} entries -> {out_dir}")

# COMMAND ----------
# MAGIC %md ### Advance watermark + write run log (versioning metadata)

# COMMAND ----------
new_watermark = result.max_last_updated or since_iso

spark.sql(f"""
    UPDATE {WATERMARK_TABLE}
    SET last_run_started_at = TIMESTAMP('{run_started_at.isoformat()}'),
        last_run_finished_at = current_timestamp(),
        last_updated_watermark = '{new_watermark}',
        status = 'SUCCESS',
        rows_ingested = {result.total_entries}
    WHERE resource_type = '{resource_type}'
""")

# Write run log via SQL insert (kept import-light: no extra Spark session churn)
for lr in log_rows:
    rid, rtype, layer, url, ts, pageno, cnt, status, err = lr
    err_sql = "NULL" if err is None else f"'{err}'"
    spark.sql(f"""
        INSERT INTO {RUN_LOG_TABLE} VALUES (
            '{rid}', '{rtype}', '{layer}', '{url.replace("'", "''")}',
            TIMESTAMP('{ts.replace('Z','')}'), {pageno}, {cnt}, '{status}', {err_sql}
        )
    """)

dbutils.jobs.taskValues.set(key="new_watermark", value=new_watermark)
dbutils.jobs.taskValues.set(key="entries_ingested", value=result.total_entries)

print(f"[{resource_type}] watermark advanced to {new_watermark}")
